# 1. 데이터 준비

In [21]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas scikit-learn matplotlib seaborn -q

import platform
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만듭니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록)
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.5.4
pandas: 3.0.6


In [22]:
# ─────────────────────────────────────────────
# 모두마켓 이번 달 주문 데이터 생성 — 자기완결적 스냅샷
# (지난 노드에서 다룬 오염 요소를 적당히 섞어 둡니다)
# ─────────────────────────────────────────────

def make_orders(seed=42, n=1500):
    np.random.seed(seed)
    regions = np.random.choice(["서울", "경기", "부산", "인천", "대구"], n, p=[0.4, 0.25, 0.15, 0.1, 0.1])
    membership = np.random.choice(["basic", "silver", "gold", "vip"], n, p=[0.5, 0.25, 0.15, 0.1])
    channels = np.random.choice(["web", "app", "app ", "APP"], n, p=[0.5, 0.4, 0.05, 0.05])
    categories = np.random.choice(["패션", "뷰티", "식품", "가전", "도서"], n)

    prices = np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 249900], n,
                            p=[0.2, 0.25, 0.2, 0.15, 0.1, 0.06, 0.04])
    quantities = np.random.choice([1, 1, 1, 2, 2, 3], n)
    amount = (prices * quantities).astype(float)

    orders = pd.DataFrame({
        "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n + 1)],
        "customer_age": np.random.normal(35, 9, n).round().astype(int),
        "region": regions,
        "membership": membership,
        "channel": channels,
        "category": categories,
        "price": prices.astype(float),
        "quantity": quantities,
        "amount": amount,
    })

    # 오염 심기: 결측·이상치·표기 혼재
    orders.loc[np.random.choice(n, 60, replace=False), "amount"] = np.nan
    orders.loc[np.random.choice(n, 30, replace=False), "customer_age"] = np.nan
    orders.loc[5, "customer_age"] = 999             # 입력 실수성 이상치
    orders.loc[8, "customer_age"] = -3              # 불가능한 음수
    orders.loc[12, "quantity"] = 80                 # 비정상적으로 큰 주문
    orders.loc[20, "region"] = " 서울 "             # 앞뒤 공백
    orders.loc[21, "region"] = "Seoul"              # 영문 표기
    orders.loc[40, "membership"] = "VIP"            # 대소문자 혼재

    return orders

orders = make_orders()
print("이번 달 주문 데이터 준비 완료:", orders.shape)
orders.head()



이번 달 주문 데이터 준비 완료: (1500, 9)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount
0,O00001,30.0,서울,silver,app,패션,19900.0,1,NaN
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0
3,O00004,24.0,경기,basic,app,뷰티,19900.0,1,NaN
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0


In [23]:
# ─────────────────────────────────────────────
#  원본 CSV 파일 준비 (실무에서는 외부에서 받아오는 단계)
# ─────────────────────────────────────────────
work_dir = Path("d006_work")
work_dir.mkdir(exist_ok=True)
input_csv  = work_dir / "orders_raw.csv"
output_csv = work_dir / "orders_clean.csv"

orders.to_csv(input_csv, index=False)
print("원본 CSV 저장:", input_csv, "—", input_csv.stat().st_size, "bytes")

원본 CSV 저장: d006_work/orders_raw.csv — 80400 bytes


# 2. 전처리 함수 작성
- 단계 함수 5개 이상을 정의, 각 함수는 한 책임만 가짐
- 인코딩(Ordinal + One-Hot)과 스케일링(Robust 등) 모두 포함

In [24]:
from sklearn.preprocessing import RobustScaler

# 단계 함수들
def load_orders(path):
    # CSV에서 주문 데이터를 읽어 옵니다.
    return pd.read_csv(path)

def clean_strings_full(df):
    # 문자열 컬럼의 공백·대소문자·표기 혼재를 정리합니다.
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )

def drop_invalid_rows(df, age_min=10, age_max=80, qty_max=10):
    # 불가능한 값과 결측을 제거합니다.
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
        .drop_duplicates()
        .reset_index(drop=True)
    )

def add_derived(df):
    # 분석용 파생 컬럼을 추가합니다.
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
        amount_class=lambda d: np.where(d["amount"] >= 100_000, "high",
                                np.where(d["amount"] >= 30_000, "mid", "low"))
    )

def encode_full(df):
    # membership=Ordinal, 나머지=One-Hot.
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    one_hots = [
        pd.get_dummies(out["region"],   prefix="region", dtype=int),
        pd.get_dummies(out["channel"],  prefix="ch",     dtype=int),
        pd.get_dummies(out["category"], prefix="cat",    dtype=int),
    ]
    return pd.concat([out] + one_hots, axis=1)

def scale_with_robust(df, cols=("customer_age", "amount", "quantity")):
    # 수치형 컬럼을 Robust로 스케일링하고 _scaled 접미사를 붙입니다.
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(scaled,
                             columns=[f"{c}_scaled" for c in cols],
                             index=df.index)
    return pd.concat([df, scaled_df], axis=1)

print("단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.")

단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.


# 3. pipeline : 함수 단위의 파이프라인 조립

In [25]:
# (심화1) verbose=True 옵션 적용
def log_step(df, step_name, verbose=False):
    # verbose일 때 현재 단계의 행·열 수를 출력하고 df를 그대로 돌려줍니다.
    if verbose:
        print(f"[{step_name:<18}] 행 {df.shape[0]:>5} | 열 {df.shape[1]:>3}")
    return df

# end-to-end 파이프라인
def preprocess(input_path, verbose=False):
    # 원본 CSV 경로를 받아 전처리된 DataFrame을 돌려줍니다.
    return (
        load_orders(input_path)
        .pipe(log_step, "load", verbose)
        .pipe(clean_strings_full)
        .pipe(log_step, "clean_strings", verbose)
        .pipe(drop_invalid_rows, age_min=10, age_max=80, qty_max=10)
        .pipe(log_step, "drop_invalid_rows", verbose)
        .pipe(add_derived)
        .pipe(log_step, "add_derived", verbose)
        .pipe(encode_full)
        .pipe(log_step, "encode", verbose)
        .pipe(scale_with_robust)
        .pipe(log_step, "scale", verbose)
    )

# 진짜 한 줄 호출
clean_df = preprocess(input_csv, verbose=True)
print("입력 행 수 → 출력 행 수:", orders.shape[0], "→", clean_df.shape[0])
print("출력 컬럼 수:", clean_df.shape[1])
clean_df.head(3)

[load              ] 행  1500 | 열   9
[clean_strings     ] 행  1500 | 열   9
[drop_invalid_rows ] 행  1404 | 열   9
[add_derived       ] 행  1404 | 열  12
[encode            ] 행  1404 | 열  25
[scale             ] 행  1404 | 열  28
입력 행 수 → 출력 행 수: 1500 → 1404
출력 컬럼 수: 28


,order_id,customer_age,region,membership,channel,category,price,quantity,amount,amount_log,...,ch_app,ch_web,cat_가전,cat_도서,cat_뷰티,cat_식품,cat_패션,customer_age_scaled,amount_scaled,quantity_scaled
0,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0,13.527430,...,1,0,1,0,0,0,0,-0.916667,10.141429,2.0
1,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0,9.898525,...,0,1,0,0,0,0,1,0.833333,-0.284286,0.0
2,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0,9.898525,...,1,0,0,0,0,0,1,0.500000,-0.284286,0.0


# 4. 결과 csv 파일 저장 및 다시 읽어 shape 유지 확인

In [26]:
# 저장 + 재로드 검증
clean_df.to_csv(output_csv, index=False)

reloaded = pd.read_csv(output_csv)
print("저장 파일:", output_csv, "—", output_csv.stat().st_size, "bytes")
print("저장 직전 shape:", clean_df.shape)
print("다시 읽은  shape:", reloaded.shape)
print("동일한가?:", clean_df.shape == reloaded.shape)

저장 파일: d006_work/orders_clean.csv — 198278 bytes
저장 직전 shape: (1404, 28)
다시 읽은  shape: (1404, 28)
동일한가?: True


# 5. 품질 리포트 함수
- 입력, 출력 행 수, 제거 비율, 결측 상위 5개 보고

In [27]:
# 품질 리포트 함수
def quality_report(before: pd.DataFrame, after: pd.DataFrame) -> dict:
    # 전처리 전·후 데이터의 품질 차이를 dict로 반환합니다.
    report = {
        "rows_before": before.shape[0],
        "rows_after":  after.shape[0],
        "removed_pct": round(100 * (1 - after.shape[0] / before.shape[0]), 2),
        "cols_before": before.shape[1],
        "cols_after":  after.shape[1],
        "new_cols":    after.shape[1] - before.shape[1],
        "missing_top5_before": (
            before.isnull().sum().sort_values(ascending=False).head(5).to_dict()
        ),
        "missing_top5_after": (
            after.isnull().sum().sort_values(ascending=False).head(5).to_dict()
        ),
        "dtypes_after": after.dtypes.value_counts().to_dict(),
    }
    return report

# 사용
rep = quality_report(orders, clean_df)
for k, v in rep.items():
    print(f"{k}: {v}")

rows_before: 1500
rows_after: 1404
removed_pct: 6.4
cols_before: 9
cols_after: 28
new_cols: 19
missing_top5_before: {'amount': 60, 'customer_age': 30, 'order_id': 0, 'membership': 0, 'region': 0}
missing_top5_after: {'order_id': 0, 'customer_age': 0, 'region': 0, 'membership': 0, 'channel': 0}
dtypes_after: {dtype('int64'): 15, dtype('float64'): 7, <StringDtype(storage='python', na_value=nan)>: 6}


# 6. 파이프라인 검증
- (심화2) 같은 파이프라인을 새 데이터 (시드 변경 또는 다른 csv)로 검증 후 결과 차이 작성

In [28]:
orders_new = make_orders(seed=7)
new_csv = work_dir / "orders_raw_seed7.csv"
orders_new.to_csv(new_csv, index=False)

clean_new = preprocess(new_csv, verbose=True)

compare = pd.DataFrame({
    "seed=42": quality_report(orders, clean_df),
    "seed=7":  quality_report(orders_new, clean_new),
}).loc[["rows_before", "rows_after", "removed_pct", "cols_after"]]
display(compare)

# 두 결과의 컬럼 구성이 같은지 확인 (빈 set이면 같음)
print("컬럼 차이:", set(clean_df.columns) ^ set(clean_new.columns))


[load              ] 행  1500 | 열   9
[clean_strings     ] 행  1500 | 열   9
[drop_invalid_rows ] 행  1401 | 열   9
[add_derived       ] 행  1401 | 열  12
[encode            ] 행  1401 | 열  25
[scale             ] 행  1401 | 열  28


,seed=42,seed=7
rows_before,1500,1500
rows_after,1404,1401
removed_pct,6.4,6.6
cols_after,28,28


컬럼 차이: set()


### 새 데이터(seed=7) 검증 결과

**1. 검증 방법**
- 데이터 생성 코드를 `make_orders(seed)` 함수로 감싸고, seed만 42에서 7로 바꿔 새 데이터를 만듦
- 수정 없이 같은 `preprocess()`를 적용하고 `quality_report()`로 두 결과를 비교함

**2. 결과 비교**

| 항목 | seed=42 | seed=7 |
|---|---|---|
| 입력 행 수 | 1500 | 1500 |
| 출력 행 수 | 1404 | 1401 |
| 제거 비율 | 6.4% | 6.6% |
| 출력 컬럼 수 | 28 | 28 |

- verbose 출력을 보면 두 데이터 모두 행은 `drop_invalid_rows` 단계에서만 줄어듦
- 컬럼 차이: `set()` → 두 결과의 컬럼 구성이 완전히 같음

**3. 차이가 생긴 이유**
- 결측·이상치를 넣는 개수는 같고(amount 60개, customer_age 30개), 들어가는 위치만 seed에 따라 달라짐
- 그래서 제거되는 행 수가 조금 달라졌지만, 제거 비율 차이는 0.2%p로 작음

**4. 결론 및 한계**
- 데이터가 바뀌어도 파이프라인이 에러 없이 동작했고, 출력 컬럼(28개)도 같았음 → 재사용 가능함을 확인
- 한계: `pd.get_dummies`는 데이터에 있는 범주로만 컬럼을 만듦. 새 데이터에 특정 지역이나 채널이 없으면 컬럼 수가 달라질 수 있음
- 한계: RobustScaler를 데이터마다 새로 fit하므로, 두 결과의 `_scaled` 값은 기준이 달라 직접 비교할 수 없음

# 7. 인코딩 전략 비교
- (심화3) One-Hot vs Target Encoding (범주별 amount 평균) 비교


In [29]:
def encode_target_mean(df, target="amount"):
    # membership=Ordinal, 나머지=범주별 target 평균으로 대체합니다.
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    for col in ["region", "channel", "category"]:
        out[f"{col}_te"] = out.groupby(col)[target].transform("mean")
    return out

base = (load_orders(input_csv)
        .pipe(clean_strings_full)
        .pipe(drop_invalid_rows)
        .pipe(add_derived))

v_onehot = encode_full(base)
v_target = encode_target_mean(base)
print("One-Hot shape:", v_onehot.shape)       # (1404, 25)
print("Target  shape:", v_target.shape)       # (1404, 16)
v_target.groupby("region")["region_te"].first()


One-Hot shape: (1404, 25)
Target  shape: (1404, 16)


region
경기    77840.588235
대구    74978.873239
부산    76732.512315
서울    72394.395797
인천    81727.027027
Name: region_te, dtype: float64

### One-Hot vs Target Encoding 비교 결과

**1. 비교 방법**
- 같은 정제 데이터(1404행·12열)에 인코딩만 다르게 적용함
- One-Hot: 범주마다 0/1 컬럼을 만듦 (`encode_full`)
- Target Encoding: 각 범주를 그 범주의 amount 평균으로 바꿈 (`encode_target_mean`)

**2. 결과**
- shape: One-Hot `(1404, 25)`, Target Encoding `(1404, 16)`
- One-Hot은 범주 수만큼 컬럼이 늘어나고, Target Encoding은 원래 컬럼 하나당 1개만 생김
- 지역별 평균 주문액 (region_te): 인천 81,727원 > 경기 77,841원 > 부산 76,733원 > 대구 74,979원 > 서울 72,394원

**3. 장단점**
- One-Hot
  - 장점: 이해하기 쉽고 데이터 누수가 없음
  - 단점: 범주가 많으면 컬럼이 너무 많아짐
- Target Encoding
  - 장점: 컬럼 수가 적고, 범주와 목표값(amount)의 관계를 숫자 하나에 담음
  - 단점: 목표값으로 만든 피처라 데이터 누수 위험이 있음 → 학습 데이터에서만 평균을 구해야 함
  - 단점: 평균이 비슷한 범주는 구분이 흐려짐 (예: 경기와 부산)

**4. 결론**
- 이 데이터는 범주가 적어서(최대 5개) **One-Hot이 적절함**
- Target Encoding은 범주가 아주 많을 때(예: 상품 ID), 누수 방지 처리와 함께 쓰는 게 좋음